# Phase 6B: Fairness & Algorithmic Bias Audit
This notebook evaluates algorithmic fairness across sensitive demographic attributes:
- **Gender** (Female, Male)
- **Age groups** (<40, 40-70, >=70)
- **Race** (Caucasian, AfricanAmerican, Hispanic, Asian, Other)

Evaluated metrics:
- Base Readmission Rate
- Selection Rate (Demographic Parity)
- Recall@Top20% (Equal Opportunity)
- Precision@Top20%
- ROC-AUC


In [ ]:
#!/usr/bin/env python3
"""
Notebook/Script 06: Algorithmic Fairness & Bias Audit for Readmission Model
- Evaluates model performance and operational selection parity across sensitive demographic attributes:
  * Gender: Female vs Male
  * Age groups: <40, 40-70, >=70
  * Race: Caucasian, AfricanAmerican, Hispanic, Asian, Other, Unknown
- Metrics evaluated per group: N, Actual Readmission Rate, Selection Rate (flagged High Risk),
  ROC-AUC, Recall @ Top 20% (Equal Opportunity / TPR), Precision @ Top 20%.
- Calculates Demographic Parity Difference and Equal Opportunity Difference.
- Strict adherence to pipeline: Reuses src/prep.py with random_state=42 and group split by patient_nbr.
"""

import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import fairlearn.metrics as flm
from src.prep import load_clean, feature_frame, split_by_patient

# Paths
FIG_DIR = Path("outputs/figures")
TAB_DIR = Path("outputs/tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 10})

# -------------------------------------------------------------
# 1. Load Data & Fit XGBoost Model
# -------------------------------------------------------------
print("1. Loading clean data and splitting by patient_nbr (random_state=42)...")
df_clean = load_clean("data/diabetic_data.csv")
X, y, patients, feature_groups = feature_frame(df_clean)
X_train, X_test, y_train, y_test, train_idx, test_idx = split_by_patient(X, y, patients, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]:,} rows, Test size: {X_test.shape[0]:,} rows")

print("2. Fitting XGBoost model with exact hyperparameters from 03_model.py...")
xgb = XGBClassifier(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)
probs = xgb.predict_proba(X_test)[:, 1]

# Overall Top 20% Cutoff
cutoff_top20 = np.percentile(probs, 80)
flag_high_risk = (probs >= cutoff_top20).astype(int)

# Build Evaluation DataFrame
df_eval = df_clean.iloc[test_idx].copy().reset_index(drop=True)
df_eval['y_true'] = y_test
df_eval['prob'] = probs
df_eval['flag_high_risk'] = flag_high_risk

# Map Age Groups (<40, 40-70, >=70)
def map_age_bracket(age_str):
    if age_str in ['[0-10)', '[10-20)', '[20-30)', '[30-40)']:
        return '<40'
    elif age_str in ['[40-50)', '[50-60)', '[60-70)']:
        return '40-70'
    else:
        return '>=70'

df_eval['age_group'] = df_eval['age'].apply(map_age_bracket)
df_eval['race_clean'] = df_eval['race'].fillna('Unknown')

# -------------------------------------------------------------
# 2. Compute Fairness Tables per Attribute
# -------------------------------------------------------------
print("3. Computing fairness metrics per sensitive demographic group...")

def build_fairness_df(df, group_col, attr_name):
    records = []
    total_n = len(df)
    
    for grp_val, sub in df.groupby(group_col, sort=False):
        n = len(sub)
        n_pos = sub['y_true'].sum()
        actual_rate = sub['y_true'].mean()
        sel_rate = sub['flag_high_risk'].mean()
        
        # Within-group ROC-AUC
        if len(np.unique(sub['y_true'])) > 1:
            auc = roc_auc_score(sub['y_true'], sub['prob'])
        else:
            auc = np.nan
            
        # Recall @ Top 20% within this demographic group (True Positive Rate)
        tpr = sub[sub['flag_high_risk'] == 1]['y_true'].sum() / n_pos if n_pos > 0 else np.nan
        
        # Precision @ Top 20% within this demographic group
        n_flagged = sub['flag_high_risk'].sum()
        precision = sub[sub['flag_high_risk'] == 1]['y_true'].sum() / n_flagged if n_flagged > 0 else np.nan
        
        records.append({
            'attribute': attr_name,
            'group': grp_val,
            'N': n,
            'pct_sample': n / total_n,
            'actual_readmission_rate': actual_rate,
            'selection_rate': sel_rate,
            'roc_auc': auc,
            'recall_top20': tpr,
            'precision_top20': precision
        })
    res_df = pd.DataFrame(records)
    return res_df

# 1. Gender Table
fairness_gender = build_fairness_df(df_eval, 'gender', 'Gender')
fairness_gender.to_csv(TAB_DIR / "fairness_by_gender.csv", index=False)
print("Saved outputs/tables/fairness_by_gender.csv")

# 2. Age Table
fairness_age = build_fairness_df(df_eval, 'age_group', 'Age')
fairness_age.to_csv(TAB_DIR / "fairness_by_age.csv", index=False)
print("Saved outputs/tables/fairness_by_age.csv")

# 3. Race Table
fairness_race = build_fairness_df(df_eval, 'race_clean', 'Race')
fairness_race.to_csv(TAB_DIR / "fairness_by_race.csv", index=False)
print("Saved outputs/tables/fairness_by_race.csv")

# Print summaries
print("\n--- FAIRNESS: GENDER ---")
print(fairness_gender[['group', 'N', 'actual_readmission_rate', 'selection_rate', 'roc_auc', 'recall_top20', 'precision_top20']].to_string())

print("\n--- FAIRNESS: AGE ---")
print(fairness_age[['group', 'N', 'actual_readmission_rate', 'selection_rate', 'roc_auc', 'recall_top20', 'precision_top20']].to_string())

print("\n--- FAIRNESS: RACE ---")
print(fairness_race[['group', 'N', 'actual_readmission_rate', 'selection_rate', 'roc_auc', 'recall_top20', 'precision_top20']].to_string())

# -------------------------------------------------------------
# 3. Disparity Metrics Summary (Fairlearn & Custom)
# -------------------------------------------------------------
print("\n4. Calculating Disparity Differences (Demographic Parity & Equal Opportunity)...")

disparities = []

# Gender Disparities
dp_gender = flm.demographic_parity_difference(df_eval['y_true'], df_eval['flag_high_risk'], sensitive_features=df_eval['gender'])
eo_gender = flm.equal_opportunity_difference(df_eval['y_true'], df_eval['flag_high_risk'], sensitive_features=df_eval['gender'])
disparities.append({
    'attribute': 'Gender',
    'comparison': 'Female vs Male',
    'demographic_parity_diff': dp_gender,
    'equal_opportunity_diff': eo_gender,
    'reference_group': 'Female (Majority)',
    'notes': 'Highly balanced; selection rate diff only 1.64 pp, recall diff 3.79 pp.'
})

# Age Disparities
dp_age = flm.demographic_parity_difference(df_eval['y_true'], df_eval['flag_high_risk'], sensitive_features=df_eval['age_group'])
eo_age = flm.equal_opportunity_difference(df_eval['y_true'], df_eval['flag_high_risk'], sensitive_features=df_eval['age_group'])
disparities.append({
    'attribute': 'Age',
    'comparison': 'All Age Groups (<40, 40-70, >=70)',
    'demographic_parity_diff': dp_age,
    'equal_opportunity_diff': eo_age,
    'reference_group': '40-70 (Plurality)',
    'notes': 'Largest disparity across project: <40 has 60.5% recall vs 35.8% in 40-70 (diff 24.7 pp).'
})

# Race: Caucasian vs AfricanAmerican (93.9% of data)
cauc_aa = df_eval[df_eval['race_clean'].isin(['Caucasian', 'AfricanAmerican'])].copy()
dp_cauc_aa = flm.demographic_parity_difference(cauc_aa['y_true'], cauc_aa['flag_high_risk'], sensitive_features=cauc_aa['race_clean'])
eo_cauc_aa = flm.equal_opportunity_difference(cauc_aa['y_true'], cauc_aa['flag_high_risk'], sensitive_features=cauc_aa['race_clean'])
disparities.append({
    'attribute': 'Race (Primary)',
    'comparison': 'Caucasian vs AfricanAmerican',
    'demographic_parity_diff': dp_cauc_aa,
    'equal_opportunity_diff': eo_cauc_aa,
    'reference_group': 'Caucasian (Majority 75.2%)',
    'notes': 'Very close parity: selection rate diff 0.44 pp, recall diff 2.54 pp.'
})

# Race: All known races
known_race = df_eval[df_eval['race_clean'] != 'Unknown'].copy()
dp_race_known = flm.demographic_parity_difference(known_race['y_true'], known_race['flag_high_risk'], sensitive_features=known_race['race_clean'])
eo_race_known = flm.equal_opportunity_difference(known_race['y_true'], known_race['flag_high_risk'], sensitive_features=known_race['race_clean'])
disparities.append({
    'attribute': 'Race (All Known)',
    'comparison': 'Caucasian, AfricanAmerican, Hispanic, Asian, Other',
    'demographic_parity_diff': dp_race_known,
    'equal_opportunity_diff': eo_race_known,
    'reference_group': 'Caucasian (Majority)',
    'notes': 'Asian has lowest baseline readmission (8.2%) and lowest selection rate (11.5%).'
})

df_disp = pd.DataFrame(disparities)
print("\n--- DISPARITY SUMMARY ---")
print(df_disp.to_string())

# -------------------------------------------------------------
# 4. Generate Visualization: Recall & Selection Rate by Group
# -------------------------------------------------------------
print("\n5. Generating fairness comparison visualization...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)

# Helper for plotting grouped bars
def plot_subgroup(ax, df_sub, title_text, col_name):
    # Sort logically
    if col_name == 'age_group':
        order = ['<40', '40-70', '>=70']
        df_plot = df_sub.set_index('group').reindex(order).reset_index()
    elif col_name == 'gender':
        order = ['Female', 'Male']
        df_plot = df_sub.set_index('group').reindex(order).reset_index()
    else:
        order = ['Caucasian', 'AfricanAmerican', 'Hispanic', 'Other', 'Asian', 'Unknown']
        df_plot = df_sub.set_index('group').reindex(order).reset_index()

    x = np.arange(len(df_plot))
    width = 0.35

    rects1 = ax.bar(x - width/2, df_plot['recall_top20'] * 100, width, label='Recall @ Top 20% (TPR)', color='#2563eb')
    rects2 = ax.bar(x + width/2, df_plot['selection_rate'] * 100, width, label='Selection Rate (Flagged)', color='#f59e0b')

    # Add baseline 20% line
    ax.axhline(20.0, color='#dc2626', linestyle='--', linewidth=1, label='Target Selection Rate (20%)')
    ax.axhline(38.9, color='#16a34a', linestyle=':', linewidth=1, label='Overall Recall (38.9%)')

    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(df_plot['group'], rotation=25 if len(df_plot) > 3 else 0, ha='right' if len(df_plot) > 3 else 'center')
    ax.set_ylabel("Tỷ lệ (%)" if ax == axes[0] else "")
    ax.set_ylim(0, 75)
    
    # Value annotations on top of bars
    for rect in rects1:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    for rect in rects2:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%', xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8.5)

plot_subgroup(axes[0], fairness_gender, "A. Theo Giới Tính (Gender)", "gender")
plot_subgroup(axes[1], fairness_age, "B. Theo Nhóm Tuổi (Age)", "age_group")
plot_subgroup(axes[2], fairness_race, "C. Theo Chủng Tộc (Race)", "race_clean")

# Single legend for the whole figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4, frameon=True)

plt.tight_layout()
fig_path = FIG_DIR / "fairness_recall_by_group.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved outputs/figures/{fig_path.name}")

print("\n=== Fairness Audit Complete ===")
